# Baebugs PA12 toy bounding-box audit

Replay of existing recorded YOLOE and OWLv2 bounding-box geometry and source identities. Not an independent inference run or species ground-truth test.

In [ ]:
import sys, json, pathlib, hashlib
root=pathlib.Path.cwd()
if not (root/'docs/toy_photo_models/results.json').exists():root=root.parent
assert (root/'site/mvp/mvp.json').exists()
sys.path.insert(0,str(root/'site/detection'))
from build_detection import compile_detection,canonical_bytes,sha
gallery=compile_detection(root)
print('graph root',gallery['source_evidence_root'],'images',len(gallery['images']))

## Verify models, coordinates and independently addressable detection identities

BBoxes refer to full-resolution source-photo pixel coordinates. Model confidence values are not independent truth labels.

In [ ]:
counts={}
for entry in gallery['images']:
    counts[entry['photo']]={m:sum(p['model']==m for p in entry['model_predictions']) for m in ('yoloe','owlv2')}
    for p in entry['model_predictions']:
        x0,y0,x1,y1=p['bbox_xyxy_pixels']
        assert 0<=x0<x1<=entry['width'] and 0<=y0<y1<=entry['height']
        data={k:p[k] for k in ('schema','source_image_sha256','source_result_sha256','model','model_identity','record_index','label','confidence','bbox_xyxy_pixels','order_inference','image_width','image_height','scientific_identity','physical_recovery')}
        assert sha(canonical_bytes(data))==p['content_sha256']
print(counts)

## Verify exact source image SHA-256 and do not promote research claims

The actual model execution occurred previously, per recorded results. This replay does not measure specimen recovery or order-classification accuracy.

In [ ]:
assert gallery['human_labels']=='NOT_VERIFIED'
assert gallery['inference_execution_this_build']=='NOT_EXECUTED'
assert gallery['physical_specimen_recovery']=='NOT_TESTED'
for entry in gallery['images']:
    source=root/entry['source_photo']['path']
    assert hashlib.sha256(source.read_bytes()).hexdigest()==entry['source_photo']['sha256']
print('PASS: complete image/source byte identity, 8 recorded detections; ground truth NOT_VERIFIED')